# Nova AI - Training Notebook

Train your own Nova Chat and Nova Code models from scratch.

**Before running:**
1. Runtime → Change runtime type → **T4 GPU** (free)
2. Add your HF token: click the 🔑 key icon on the left → add secret `HF_TOKEN`
3. Change `YOUR_HF_USERNAME` in the Push cell to your Hugging Face username
4. Run all cells top to bottom

**What this trains:** Two real GPT models (124M params each) from scratch.
No pretrained weights. Your data. Your weights.

In [ ]:
# ── Cell 1: Check GPU ──────────────────────────────────────────────
import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), 'GB')
else:
    print('WARNING: No GPU found. Go to Runtime → Change runtime type → T4 GPU')

In [ ]:
# ── Cell 2: Clone your repo and install dependencies ───────────────
!git clone https://github.com/genity5-collab/Phollama-OS.git
%cd Phollama-OS
!git pull

!pip install torch tokenizers datasets huggingface_hub numpy -q
print('Dependencies installed.')

In [ ]:
# ── Cell 3: Prepare data and train tokenizer ───────────────────────
import os
os.makedirs('data',   exist_ok=True)
os.makedirs('models', exist_ok=True)

!python data/prepare.py
print('Data ready.')

In [ ]:
# ── Cell 4: Train Nova Chat ────────────────────────────────────────
# This takes ~30-60 min on a free T4 GPU
# Watch: train loss and val loss both going DOWN = good
# If val loss goes UP while train loss goes DOWN = overfitting, stop

!python src/train.py --model chat
print('Nova Chat training complete.')

In [ ]:
# ── Cell 5: Train Nova Code ────────────────────────────────────────
!python src/train.py --model code
print('Nova Code training complete.')

In [ ]:
# ── Cell 6: Test the models ────────────────────────────────────────
import sys
sys.path.append('.')
import torch
from src.model import NovaGPT
from src.tokenizer import NovaTokenizer
from src.config import NovaChatConfig, NovaCodeConfig

tokenizer = NovaTokenizer('models/nova_tokenizer.json')

def test_model(config, prompt, model_type):
    path = f'checkpoints/nova-{model_type}/best.pt'
    if not os.path.exists(path):
        path = f'checkpoints/nova-{model_type}/final.pt'
    model, _ = NovaGPT.load_checkpoint(path, config)
    model.eval()
    ids = tokenizer.encode(prompt, add_special_tokens=False)
    idx = torch.tensor([ids])
    with torch.no_grad():
        out = model.generate(idx, max_new_tokens=100, temperature=0.8, top_k=40)
    new_ids  = out[0][len(ids):].tolist()
    response = tokenizer.decode(new_ids)
    for stop in ['<|user|>', '<|system|>', '<eos>']:
        if stop in response:
            response = response[:response.index(stop)]
    return response.strip()

# Test Nova Chat
chat_prompt = '<|system|>You are Nova.<|user|>What is your name?<|assistant|>'
print('Nova Chat:')
print(test_model(NovaChatConfig, chat_prompt, 'chat'))

# Test Nova Code
code_prompt = '<|system|>You are Nova Code.<|user|>Write hello world in Python.<|assistant|>'
print('\nNova Code:')
print(test_model(NovaCodeConfig, code_prompt, 'code'))

In [ ]:
# ── Cell 7: Push to Hugging Face ───────────────────────────────────
# Change this to your username:
YOUR_HF_USERNAME = 'genity5-collab'  # CHANGE THIS

from google.colab import userdata
from huggingface_hub import login, HfApi
import shutil

login(userdata.get('HF_TOKEN'))
api = HfApi()

for model_type in ['chat', 'code']:
    repo_id  = f'{YOUR_HF_USERNAME}/nova-{model_type}-124m'
    best_pt  = f'checkpoints/nova-{model_type}/best.pt'
    final_pt = f'checkpoints/nova-{model_type}/final.pt'
    tok_path = 'models/nova_tokenizer.json'

    # Create repo if it doesn't exist
    try:
        api.create_repo(repo_id=repo_id, exist_ok=True)
    except Exception as e:
        print(f'Repo already exists or error: {e}')

    # Upload files
    for local, remote in [
        (best_pt  if os.path.exists(best_pt) else final_pt, 'model.pt'),
        (tok_path, 'tokenizer.json'),
        ('src/model.py',   'src/model.py'),
        ('src/config.py',  'src/config.py'),
        ('src/tokenizer.py', 'src/tokenizer.py'),
        ('src/generate.py', 'src/generate.py'),
    ]:
        if os.path.exists(local):
            api.upload_file(
                path_or_fileobj=local,
                path_in_repo=remote,
                repo_id=repo_id,
            )
            print(f'Uploaded: {remote} -> {repo_id}')

    print(f'\nNova {model_type} live at: https://huggingface.co/{repo_id}')

print('\nAll models pushed to Hugging Face!')

In [ ]:
# ── Cell 8: Download checkpoints to your PC ────────────────────────
# This downloads the trained weights so you can run them locally
from google.colab import files

for model_type in ['chat', 'code']:
    best  = f'checkpoints/nova-{model_type}/best.pt'
    final = f'checkpoints/nova-{model_type}/final.pt'
    path  = best if os.path.exists(best) else final
    if os.path.exists(path):
        files.download(path)
        print(f'Downloading: {path}')

files.download('models/nova_tokenizer.json')
print('Download complete. Save these files to your nova-ai/models/ folder.')

## After downloading

1. Save the `.pt` files to `checkpoints/nova-chat/best.pt` and `checkpoints/nova-code/best.pt` on your PC
2. Save `nova_tokenizer.json` to `models/nova_tokenizer.json`
3. Run chat locally:
   ```
   python src/generate.py --model chat
   python src/generate.py --model code
   ```